<div dir="rtl">

# کارگاه جلسه ۲۵ — ارزیابی مدل‌های طبقه‌بندی

نوشتن این کدها در امتحان پرسیده نمی‌شود.

## تمرین عملی کلاس (۳۰ دقیقه): ریزش مشتری مخابرات

داده Telco Customer Churn (همان داده جزوه) را بارگذاری کنید. فرض کنید پیشنهاد تخفیف به هر مشتری (چه واقعاً برود چه نه) **۱ واحد** و از دست دادن مشتری ریزشی‌ای که پیشنهاد نگرفته **۵ واحد** هزینه دارد.

1. با تقسیم طبقه‌بندی‌شده ۲۰ درصدی (`random_state=42`)، این مدل‌ها را با `StratifiedKFold` پنج‌بخشی و معیارهای F1، میانگین دقت و سطح زیر ROC مقایسه کنید: مدل پایه، رگرسیون لجستیک، 25-NN، جنگل تصادفی با وزن‌دهی متوازن و HistGradientBoosting.
2. برای رگرسیون لجستیک، C و وزن‌دهی کلاس را با `GridSearchCV` و معیار میانگین دقت تنظیم کنید.
3. آستانه نظری هزینه‌محور چیست؟ با پیش‌بینی‌های خارج از بخش داده آموزش، آستانه‌ای را که هزینه کل را کمینه می‌کند پیدا کنید.
4. مدل نهایی را روی آزمون با آستانه ۰.۵ و آستانه منتخب بسنجید: ماتریس درهم‌ریختگی و هزینه کل.
5. (اختیاری) اگر `imbalanced-learn` نصب است، SMOTE را داخل Pipeline آن امتحان کنید.

</div>

In [1]:
# Write your solution here

<div dir="rtl">

### پاسخ تمرین

</div>

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_validate, GridSearchCV,
                                     cross_val_predict)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(url)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")
y = (telco["Churn"] == "Yes").astype(int)
X = telco.drop(columns=["customerID", "Churn"])
num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat = [c for c in X.columns if c not in num]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1. Model comparison
models = {"Baseline": DummyClassifier(), "Logistic": LogisticRegression(max_iter=2000),
          "25-NN": KNeighborsClassifier(25),
          "Forest, balanced": RandomForestClassifier(300, min_samples_leaf=5, class_weight="balanced",
                                                     random_state=42, n_jobs=-1),
          "HistGB": HistGradientBoostingClassifier(random_state=42)}
rows = []
for name, clf in models.items():
    r = cross_validate(make_pipeline(preprocess, clf), X_train, y_train, cv=cv,
                       scoring=["f1", "average_precision", "roc_auc"])
    rows.append({"model": name, "F1": r["test_f1"].mean(), "AP": r["test_average_precision"].mean(),
                 "ROC_AUC": r["test_roc_auc"].mean()})
print(pd.DataFrame(rows).set_index("model").round(3))

# 2. Tune logistic regression
gs = GridSearchCV(make_pipeline(preprocess, LogisticRegression(max_iter=2000)),
                  {"logisticregression__C": [0.01, 0.1, 1, 10],
                   "logisticregression__class_weight": [None, "balanced"]},
                  cv=cv, scoring="average_precision").fit(X_train, y_train)
print(f"best={ {k.split('__')[1]: v for k, v in gs.best_params_.items()} }  CV AP={gs.best_score_:.3f}")

# 3. Cost-based threshold from out-of-fold probabilities (FP costs 1, FN costs 5)
c_fp, c_fn = 1, 5
print(f"theoretical threshold={c_fp / (c_fp + c_fn):.3f}")
p_oof = cross_val_predict(gs.best_estimator_, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
def cost(y_true, p, t):
    tn, fp, fn, tp = confusion_matrix(y_true, p >= t).ravel()
    return c_fp * fp + c_fn * fn
thresholds = np.arange(0.05, 0.95, 0.01)
costs = [cost(y_train, p_oof, t) for t in thresholds]
best_t = thresholds[int(np.argmin(costs))]
print(f"OOF cost at 0.5={cost(y_train, p_oof, 0.5)}  best threshold={best_t:.2f}  OOF cost={min(costs)}")

# 4. Test set, once
p_test = gs.predict_proba(X_test)[:, 1]
for t in [0.5, best_t]:
    tn, fp, fn, tp = confusion_matrix(y_test, p_test >= t).ravel()
    print(f"threshold={t:.2f}  TN={tn} FP={fp} FN={fn} TP={tp}  total cost={c_fp * fp + c_fn * fn}")

                     F1     AP  ROC_AUC
model                                  
Baseline          0.000  0.265    0.500
Logistic          0.592  0.661    0.846
25-NN             0.602  0.611    0.832
Forest, balanced  0.631  0.654    0.846
HistGB            0.565  0.650    0.838


best={'C': 1, 'class_weight': None}  CV AP=0.661
theoretical threshold=0.167


OOF cost at 0.5=3847  best threshold=0.16  OOF cost=2393
threshold=0.50  TN=926 FP=109 FN=165 TP=209  total cost=934
threshold=0.16  TN=612 FP=423 FN=37 TP=337  total cost=608


<div dir="rtl">

### (اختیاری) SMOTE با imbalanced-learn

```python
# pip install imbalanced-learn
# from imblearn.pipeline import make_pipeline as make_imb_pipeline
# from imblearn.over_sampling import SMOTE
# model = make_imb_pipeline(preprocess, SMOTE(random_state=42), LogisticRegression(max_iter=2000))
# cross_validate(model, X_train, y_train, cv=cv, scoring=["f1", "average_precision"])
```

در Pipeline کتابخانه imbalanced-learn، گام SMOTE فقط هنگام `fit` (روی بخش آموزش هر دور) اجرا می‌شود و بخش اعتبارسنجی دست‌نخورده می‌ماند.

</div>

<div dir="rtl">

### پاسخ‌های کلیدی

- مقایسه (اعتبارسنجی ۵ بخشی): مدل پایه F1 صفر و میانگین دقت ۰.۲۶۵؛ رگرسیون لجستیک F1 ۰.۵۹۲ و میانگین دقت **۰.۶۶۱**؛ 25-NN ۰.۶۰۲ و ۰.۶۱۱؛ جنگل متوازن بهترین F1 (**۰.۶۳۱**) با میانگین دقت ۰.۶۵۴؛ HistGB ۰.۵۶۵ و ۰.۶۵۰. سطح زیر ROC همه حدود ۰.۸۳ تا ۰.۸۵ است: کیفیت رتبه‌بندی تقریباً یکسان، اما F1 در آستانه ۰.۵ متفاوت.
- تنظیم لجستیک: C برابر ۱ و **بدون وزن‌دهی** (میانگین دقت ۰.۶۶۱). وزن‌دهی رتبه‌بندی را بهتر نمی‌کند؛ اثرش مانند جابه‌جایی آستانه است که در گام بعد مستقیم انجام می‌دهیم.
- آستانه نظری یک تقسیم بر شش، یعنی **۰.۱۶۷** است. کمینه هزینه روی پیش‌بینی‌های خارج از بخش در آستانه **۰.۱۶** است (هزینه ۲۳۹۳ در برابر ۳۸۴۷ در آستانه ۰.۵)؛ بسیار نزدیک آستانه نظری، یعنی احتمال‌های لجستیک خوب کالیبره‌اند.
- روی آزمون: با آستانه ۰.۵، ۱۶۵ مشتری ریزشی از دست می‌روند و هزینه کل **۹۳۴** است. با آستانه ۰.۱۶ فقط ۳۷ مشتری از دست می‌روند (به قیمت ۴۲۳ پیشنهاد اضافی) و هزینه کل **۶۰۸** است: حدود ۳۵ درصد صرفه‌جویی بدون آموزش دوباره مدل.

</div>